# Understanding the HABC Firedrake Solver (`habc.py`)

This notebook explains the structure and numerical ideas behind the solver in `habc.py`.

It focuses on:
- the acoustic wave equation discretization used in the code,
- how the hybrid absorbing boundary condition (HABC) is built with submeshes,
- how the implementation maps to each function in the module.

## References used for structure

This notebook is inspired by the didactic style of:
- Devito HABC notebook: https://github.com/devitocodes/devito/blob/main/examples/seismic/abc_methods/04_habc.ipynb
- Firedrake FWI demo: https://www.firedrakeproject.org/demos/full_waveform_inversion.py.html

The text below is tailored to your current `habc.py` implementation.

In [2]:
from importlib.util import module_from_spec, spec_from_file_location
from pathlib import Path
import inspect
import numpy as np

repo_dir = Path.cwd()
habc_path = repo_dir / "habc.py"
assert habc_path.exists(), f"Could not find {habc_path}"

spec = spec_from_file_location("habc_solver", habc_path)
habc = module_from_spec(spec)
spec.loader.exec_module(habc)

print("Loaded module from:", habc_path.resolve())

AttributeError: 'NoneType' object has no attribute '__dict__'

In [ ]:
key_symbols = [
    "SimulationConfig",
    "ricker_wavelet",
    "create_submeshes",
    "build_source_term",
    "build_transition_weights",
    "wave_equation_solver",
    "apply_habc_blending",
    "main",
]

for name in key_symbols:
    obj = getattr(habc, name)
    signature = str(inspect.signature(obj)) if callable(obj) else "(not callable)"
    print(f"{name}{signature}")

## 1) PDE and time discretization used

The code solves the acoustic wave equation on the parent mesh with

$$m = rac{1}{c^2}, uad martial_{tt}u - 
abla^2 u = f_s(x,t).$$

Using second-order time stepping, the parent-domain term implemented in `wave_equation_solver` is equivalent to

$$mrac{u^{n+1} - 2u^n + u^{n-1}}{elta t^2}.$$

The source is represented as a dual-space object (`Cofunction`) built from a `VertexOnlyMesh`, exactly as in Firedrake wave/FWI examples.

## 2) HABC idea in this implementation

The hybrid absorbing treatment is implemented with three submeshes (left, right, top):

1. Solve one-way wave-like updates on each submesh (with sign/direction per boundary).
2. Enforce interface consistency using `EquationBC` between submesh unknowns and parent-trace values.
3. Blend submesh updates back into the parent field using transition weights.

Per-submesh blend in code (`apply_habc_blending`) is:

$$u_0^{n+1} eftarrow (1-w_i)u_0^{n+1} + w_i u_i^{n+1}.$$

where each $w_i$ decays from the boundary layer into the interior.

## 3) Function-by-function map

| Function | Role in solver |
|---|---|
| `SimulationConfig` | Centralizes mesh, timestep, source, and marker parameters. |
| `mark_submesh` + `create_submeshes` | Builds absorbing subdomains and intersected measure `dx0`. |
| `build_source_term` | Constructs point-source cofunction in parent dual space. |
| `build_transition_weights` | Creates blending weights on left/right/top submeshes. |
| `wave_equation_solver` | Assembles mixed variational problem + solver + BC coupling. |
| `apply_habc_blending` | Merges submesh contributions into parent update each step. |
| `main` | Orchestrates setup, time loop, source injection, solve, blend, output. |

In [ ]:
# Quick look at docstrings for the most important solver functions
for name in ["wave_equation_solver", "apply_habc_blending", "main"]:
    obj = getattr(habc, name)
    print(f"\n{name}:")
    print(inspect.getdoc(obj))

In [ ]:
# Inspect the source wavelet used in forcing
times = np.linspace(0.0, 0.3, 301)
values = np.array([habc.ricker_wavelet(t, peak_frequency=7.0) for t in times])

print("Ricker stats:")
print("  min =", values.min())
print("  max =", values.max())
print("  argmax time =", times[values.argmax()])

try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(8, 3))
    plt.plot(times, values)
    plt.title("Ricker wavelet used by habc.py")
    plt.xlabel("time [s]")
    plt.ylabel("amplitude")
    plt.grid(True, alpha=0.3)
    plt.show()
except Exception as exc:
    print("Matplotlib unavailable, skipping plot:", exc)

## 4) Optional: run a lightweight demo

This cell runs the same `main` function with a smaller configuration so the workflow is easier to test interactively.

Set `RUN_DEMO = True` to execute.

In [ ]:
from time import perf_counter

demo_cfg = habc.SimulationConfig(
    dt=0.001,
    final_time=0.02,
    nx=30,
    ny=30,
    progress_interval=10,
)

print(demo_cfg)

RUN_DEMO = False
if RUN_DEMO:
    t0 = perf_counter()
    habc.main(demo_cfg)
    print(f"Demo completed in {perf_counter() - t0:.2f} s")
else:
    print("Set RUN_DEMO = True to execute the solver.")

In [ ]:
output = Path("solution.pvd")
if output.exists():
    print("Output found:", output.resolve())
else:
    print("No output file yet. Run demo cell or execute `python3 habc.py`.")

## 5) Suggested experiments

To better understand behavior, vary one parameter at a time:
- Increase/decrease `boundary_distance` to test absorbing-layer thickness.
- Change `frequency_peak` in `SimulationConfig` and compare stability/dispersion behavior.
- Increase `nx`, `ny` while keeping CFL-like timing conservative.
- Try multiple source locations by increasing `ensemble_size` and `source_locations`.

These experiments follow the same philosophy used in the Devito and Firedrake tutorial references.